This notebook extends the ["Agents"]("../../LangChain Agents & Tools/Labs/Agents.ipynb") example.

In [ ]:
!pip install -q langchain langchain-openai langgraph-checkpoint-sqlite pandas

In [ ]:
import sqlite3
import pandas as pd

from google.colab import userdata
from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver
from pydantic import SecretStr
from typing import List


api_key = SecretStr(userdata.get('OPENAI_API_KEY'))


def print_conversation(conversation: List[AIMessage]):
    for message in conversation:
        message.pretty_print()

In [ ]:
sqlite_connection = sqlite3.connect("/content/checkpointer.db", check_same_thread=False)
checkpointer = SqliteSaver(sqlite_connection)

In [ ]:
def explore_database():
    tables_df = pd.read_sql_query("SELECT * FROM sqlite_master WHERE type='table';", sqlite_connection)

    result = {}
    for table_name in tables_df["name"]:
        result[table_name] = pd.read_sql_query(f"SELECT * FROM \"{table_name}\"", sqlite_connection)

    return result

In [ ]:
@tool
def lookup_tour_stop(artist: str) -> str:
    """
    Look up the next city and venue for an artist from a small curated tour calendar.

    Args:
        artist: The artist or band name to search for.
    """

    stops = {
        "breaking benjamin": "Sofia - Arena 8888",
        "placido domingo": "Varna - Palace of Culture and Sports",
        "vassil petrov & jp3": "Shumen - City Stage",
    }
    return stops.get(artist.strip().lower(), "Could not find any tour stops.")

@tool
def estimate_drive_time(origin: str, destination: str) -> str:
    """
    Estimate drive time between cities in Bulgaria.

    Args:
        origin: The departure city.
        destination: The arrival city.
    """
    routes = {
        ("plovdiv", "sofia"): "About 1 hour and 45 minutes.",
        ("shumen", "varna"): "About 1 hour.",
        ("plovdiv", "shumen"): "About 2 hours and 30 minutes."
    }
    key = (origin.strip().lower(), destination.strip().lower())
    return routes.get(key, "Could not estimate the drive time.")

In [ ]:
agent = create_agent(
    model=ChatOpenAI(model="gpt-5-nano", api_key=api_key),
    tools=[lookup_tour_stop, estimate_drive_time],
    system_prompt="You are a practical live-music concierge. Use tools when they help you give a precise answer.",
    checkpointer=checkpointer
)

In [ ]:
config = {
    "configurable": {
        "thread_id": "1"
    }
}

In [ ]:
plan_concert_trip = agent.invoke(
    input={
      "messages": HumanMessage("I'm in Plovdiv on Friday and want to hear live jazz without wasting the whole evening on travel. Check the current mini tour for \"Vassil Petrov & JP3\", figure out the relevant venue, and estimate the drive time.")
    },
    config=config
)

In [ ]:
print_conversation(plan_concert_trip["messages"])

In [ ]:
first_exploration = explore_database()

In [ ]:
first_exploration["checkpoints"]

In [ ]:
first_exploration["writes"]

In [ ]:
test_agent_memory = agent.invoke(
    input={
        "messages": [HumanMessage("What do you remember about me?")]
    },
    config=config
)

In [ ]:
print_conversation(test_agent_memory["messages"])

In [ ]:
second_exploration = explore_database()

In [ ]:
second_exploration["checkpoints"]

In [ ]:
second_exploration["writes"]